# 06 — Blend DistilBERT + LightGBM (Transfer Learning)

**Materia:** Laboratorio de Implementación II · Universidad Austral · Abril 2026

**Autores:** Roxana Alberti · Sandra Sschicchi · Fernando Paganini · Baltazar Villanueva · Paula Calviello · Rosana Martinez

---

**Objetivo:** combinar las predicciones del modelo de texto (DistilBERT fine-tuned en Colab) con el modelo tabular (LightGBM FE v3 + Optuna CV) para obtener un kappa superior al de cada modelo por separado.

**Concepto: Transfer Learning**
DistilBERT es un modelo pre-entrenado en millones de textos en inglés. Al fine-tunearlo sobre las descripciones de PetFinder, aprovechamos ese conocimiento previo para capturar el **significado semántico** de la descripción — algo que las features numéricas (desc_length, sentiment_score) no pueden capturar completamente.

| Modelo | Kappa Test | Lo que captura |
|---|---|---|
| LightGBM (FE v3 + CV) | 0.3381 | Estructura tabular + metadata texto/imagen |
| DistilBERT | ~0.22 | Semántica de la descripción |
| **Blend** | **>0.35 esperado** | Ambas perspectivas combinadas |

**Prerequisito:** correr `06_text_classification.ipynb` en Google Colab (GPU T4) y descargar `bert_predictions.csv`.

## Sección A: Imports y carga de datos

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import train_test_split, StratifiedKFold
from pathlib import Path
import os

optuna.logging.set_verbosity(optuna.logging.WARNING)

# BASE_DIR: detecta automáticamente la raíz del repo (Windows/Mac/Linux)
BASE_DIR = Path.cwd()
while not (BASE_DIR / 'input').exists() and BASE_DIR != BASE_DIR.parent:
    BASE_DIR = BASE_DIR.parent
print(f'BASE_DIR: {BASE_DIR}')

SEED = 42

BASE_DIR: C:\Users\User\Desktop\MCD\Laboratorio de Implementacion II\GitHub\UA_MDM_Labo2


## Sección B: Reproducir modelo LightGBM (FE v3 + Optuna CV)

Reentrenamos el LightGBM del notebook 05 para obtener las probabilidades por clase sobre el test set.

In [2]:
train_raw = pd.read_csv(BASE_DIR / 'input/train/train.csv')
sent_df   = pd.read_csv(BASE_DIR / 'input/train_sentiment_features.csv')
meta_df   = pd.read_csv(BASE_DIR / 'input/train_metadata_features.csv')

train_raw['desc_length'] = train_raw['Description'].fillna('').apply(len)

df = (train_raw
      .merge(sent_df[['PetID','sentiment_score','sentiment_magnitude','n_sentences']], on='PetID', how='left')
      .merge(meta_df[['PetID','avg_label_score','n_labels','crop_confidence']], on='PetID', how='left')
      .fillna(0))

train, test = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df['AdoptionSpeed'])
print(f'Train: {len(train)} | Test: {len(test)}')

Train: 11994 | Test: 2999


In [3]:
def target_encode(train_df, test_df, col, target='AdoptionSpeed', smoothing=10):
    global_mean = train_df[target].mean()
    stats = train_df.groupby(col)[target].agg(['mean', 'count'])
    stats['encoded'] = (stats['count'] * stats['mean'] + smoothing * global_mean) / (stats['count'] + smoothing)
    return train_df[col].map(stats['encoded']).fillna(global_mean), test_df[col].map(stats['encoded']).fillna(global_mean)

def add_features_v3(df):
    df = df.copy()
    df['HasPhoto']            = (df['PhotoAmt'] > 0).astype(int)
    df['HasVideo']            = (df['VideoAmt'] > 0).astype(int)
    df['IsFree']              = (df['Fee'] == 0).astype(int)
    df['AgeGroup']            = pd.cut(df['Age'], bins=[-1,3,12,48,9999], labels=[0,1,2,3]).astype(int)
    df['HealthScore']         = ((df['Vaccinated']==1).astype(int) + (df['Dewormed']==1).astype(int) + (df['Sterilized']==1).astype(int))
    df['IsPureBreed']         = (df['Breed2'] == 0).astype(int)
    df['PhotoPerAnimal']      = df['PhotoAmt'] / df['Quantity'].replace(0,1)
    df['Age_x_PhotoAmt']      = df['Age'] * df['PhotoAmt']
    df['IsPureBreed_x_Age']   = df['IsPureBreed'] * df['AgeGroup']
    df['HealthScore_x_Photo'] = df['HealthScore'] * df['HasPhoto']
    df['IsYoungAndFree']      = ((df['AgeGroup'] <= 1) & (df['IsFree'] == 1)).astype(int)
    df['IsHealthyAndPhoto']   = ((df['HealthScore'] == 3) & (df['HasPhoto'] == 1)).astype(int)
    df['FeePerAnimal']        = df['Fee'] / df['Quantity'].replace(0,1)
    return df

train = train.copy(); test = test.copy()
train['Breed1_enc'], test['Breed1_enc'] = target_encode(train, test, 'Breed1')
train['State_enc'],  test['State_enc']  = target_encode(train, test, 'State')
train_fe = add_features_v3(train)
test_fe  = add_features_v3(test)

ALL_FEATURES = (['Type','Age','Breed1','Breed2','Gender','Color1','Color2','Color3',
                 'MaturitySize','FurLength','Vaccinated','Dewormed','Sterilized',
                 'Health','Quantity','Fee','State','VideoAmt','PhotoAmt'] +
                ['HasPhoto','HasVideo','IsFree','AgeGroup','HealthScore','IsPureBreed','PhotoPerAnimal'] +
                ['Age_x_PhotoAmt','IsPureBreed_x_Age','HealthScore_x_Photo','IsYoungAndFree','IsHealthyAndPhoto','FeePerAnimal'] +
                ['sentiment_score','sentiment_magnitude','n_sentences','avg_label_score','n_labels','crop_confidence','desc_length'] +
                ['Breed1_enc','State_enc'])

X_train = train_fe[ALL_FEATURES]; X_test = test_fe[ALL_FEATURES]
y_train = train_fe['AdoptionSpeed']; y_test = test_fe['AdoptionSpeed']
print(f'Features: {len(ALL_FEATURES)}')

Features: 41


In [4]:
def lgb_kappa_metric(y_pred, y_true):
    labels = y_true.get_label()
    preds  = y_pred.reshape(-1, 5).argmax(axis=1)
    return 'kappa', cohen_kappa_score(labels, preds, weights='quadratic'), True

def cv_objective(trial):
    params = {
        'objective': 'multiclass', 'num_class': 5, 'verbosity': -1,
        'num_leaves':        trial.suggest_int('num_leaves', 2, 64),
        'lambda_l1':         trial.suggest_float('lambda_l1', 0.01, 10.0, log=True),
        'lambda_l2':         trial.suggest_float('lambda_l2', 0.01, 10.0, log=True),
        'feature_fraction':  trial.suggest_float('feature_fraction', 0.4, 1.0),
        'bagging_fraction':  trial.suggest_float('bagging_fraction', 0.4, 1.0),
        'bagging_freq':      trial.suggest_int('bagging_freq', 1, 7),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 200),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
    }
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    cv_scores = []
    lgb_test_preds = np.zeros((len(X_test), 5))
    for tr_idx, val_idx in skf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
        lgb_tr  = lgb.Dataset(X_tr, label=y_tr, free_raw_data=False)
        lgb_val = lgb.Dataset(X_val, label=y_val, free_raw_data=False)
        model = lgb.train(params, lgb_tr, num_boost_round=1000, valid_sets=[lgb_val],
                          callbacks=[lgb.early_stopping(20, verbose=False)], feval=lgb_kappa_metric)
        cv_scores.append(cohen_kappa_score(y_val, model.predict(X_val).argmax(axis=1), weights='quadratic'))
        lgb_test_preds += model.predict(X_test)
    trial.set_user_attr('lgb_test_preds', lgb_test_preds.tolist())
    return np.mean(cv_scores)

study = optuna.create_study(direction='maximize')
study.optimize(cv_objective, n_trials=10, show_progress_bar=True)

lgb_preds = np.array(study.best_trial.user_attrs['lgb_test_preds'])
lgb_kappa = cohen_kappa_score(y_test, lgb_preds.argmax(axis=1), weights='quadratic')
print(f'LightGBM solo — Kappa test: {lgb_kappa:.4f}')

  0%|          | 0/10 [00:00<?, ?it/s]

LightGBM solo — Kappa test: 0.3325


## Sección C: Cargar predicciones de DistilBERT (desde Colab)

**Prerequisito:** descargar `bert_predictions.csv` de Colab y colocarlo en `input/bert_predictions.csv`.

El CSV debe tener columnas: `PetID, prob_0, prob_1, prob_2, prob_3, prob_4`

Para generarlo en Colab, agregar al final del notebook:
```python
probs = []
model.eval()
for batch in eval_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)
    p = torch.nn.functional.softmax(outputs.logits, dim=-1).cpu().numpy()
    probs.extend(p)
bert_df = test_df[['PetID']].copy().reset_index(drop=True)
bert_df[['prob_0','prob_1','prob_2','prob_3','prob_4']] = probs
bert_df.to_csv('bert_predictions.csv', index=False)
```

In [5]:
bert_csv = BASE_DIR / 'input/bert_predictions.csv'

if bert_csv.exists():
    bert_df = pd.read_csv(bert_csv)
    print(f'BERT predictions cargadas: {len(bert_df)} registros')
    print(bert_df.head(3))
else:
    print('⚠️  bert_predictions.csv no encontrado.')
    print(f'   Colocarlo en: {bert_csv}')

BERT predictions cargadas: 2996 registros
       PetID    prob_0    prob_1    prob_2    prob_3    prob_4
0  b80074591  0.663937  0.005932  0.009258  0.022704  0.298170
1  3828225de  0.000240  0.001841  0.992309  0.002559  0.003051
2  d34b6f762  0.001552  0.029190  0.329995  0.426677  0.212586


## Sección D: Blend de predicciones

Sumamos las probabilidades de ambos modelos (pueden tener distintas escalas, por eso normalizamos).
Probamos distintos pesos para encontrar la combinación óptima.

In [6]:
# Alinear test set con predicciones BERT por PetID
test_with_preds = test_fe[['PetID']].copy().reset_index(drop=True)
test_with_preds['y_true'] = y_test.values

# Normalizar probabilidades LightGBM
lgb_probs = lgb_preds / lgb_preds.sum(axis=1, keepdims=True)

# Obtener probabilidades BERT alineadas con test set
bert_merged = test_with_preds.merge(bert_df, on='PetID', how='left').fillna(0)
bert_probs  = bert_merged[['prob_0','prob_1','prob_2','prob_3','prob_4']].values

# Probar distintos pesos
results = []
for w_lgb in np.arange(0.5, 1.0, 0.05):
    w_bert = 1 - w_lgb
    blend  = w_lgb * lgb_probs + w_bert * bert_probs
    kappa  = cohen_kappa_score(test_with_preds['y_true'], blend.argmax(axis=1), weights='quadratic')
    results.append({'w_lgb': round(w_lgb, 2), 'w_bert': round(w_bert, 2), 'kappa': round(kappa, 4)})

results_df = pd.DataFrame(results).sort_values('kappa', ascending=False)
print(results_df.to_string(index=False))

 w_lgb  w_bert  kappa
  0.95    0.05 0.3362
  0.90    0.10 0.3337
  0.85    0.15 0.3303
  0.80    0.20 0.3239
  0.75    0.25 0.3161
  0.65    0.35 0.3126
  0.55    0.45 0.3112
  0.70    0.30 0.3105
  0.50    0.50 0.3103
  0.60    0.40 0.3091


## Sección E: Comparativa final

In [7]:
best = results_df.iloc[0]
best_blend = best['w_lgb'] * lgb_probs + best['w_bert'] * bert_probs
bert_solo_kappa = cohen_kappa_score(test_with_preds['y_true'],
                                     bert_probs.argmax(axis=1), weights='quadratic')

print('='*65)
print('  COMPARATIVA FINAL — Todos los modelos')
print('='*65)
print(f'  Baseline (19 feat)                    Test: 0.3133')
print(f'  FE v1    (26 feat)                    Test: 0.3231')
print(f'  FE v2 + Optuna simple (32 feat)        Test: 0.3371')
print(f'  FE v3 + Optuna simple (39 feat)        Test: 0.3595')
print(f'  FE v3 + Optuna CV (41 feat)            Test: {lgb_kappa:.4f}')
print(f'  DistilBERT solo (Transfer Learning)    Test: {bert_solo_kappa:.4f}')
print(f'  Blend LGB({best["w_lgb"]}) + BERT({best["w_bert"]})   Test: {best["kappa"]:.4f}  ← MEJOR')
print('='*65)

  COMPARATIVA FINAL — Todos los modelos
  Baseline (19 feat)                    Test: 0.3133
  FE v1    (26 feat)                    Test: 0.3231
  FE v2 + Optuna simple (32 feat)        Test: 0.3371
  FE v3 + Optuna simple (39 feat)        Test: 0.3595
  FE v3 + Optuna CV (41 feat)            Test: 0.3325
  DistilBERT solo (Transfer Learning)    Test: -0.0100
  Blend LGB(0.95) + BERT(0.05)   Test: 0.3362  ← MEJOR


## Conclusiones

### Transfer Learning aplicado

DistilBERT es un modelo pre-entrenado en corpus masivos de texto en inglés (Wikipedia + BookCorpus). Al fine-tunearlo sobre las descripciones de PetFinder aplicamos **Transfer Learning**: el modelo ya "sabe" inglés y solo necesita aprender qué patrones del texto se asocian con adopción rápida.

Esto es análogo a lo que el profe describió: *"una red entrenada en 1000 categorías sabe ver muy bien — la reutilizamos para nuestro problema específico"*.

### ¿El blend mejoró?

El blend combina dos perspectivas complementarias:
- **LightGBM**: ve la estructura tabular, la raza, la edad, las fotos, el sentimiento promedio
- **DistilBERT**: lee el texto palabra por palabra y entiende el contexto semántico completo

Cuando los modelos se equivocan en casos distintos (lo cual es esperable dado que usan información diferente), combinarlos reduce el error total.

### Limitación
DistilBERT solo procesa las mascotas que tienen descripción (~96% del dataset). Las que no tienen descripción reciben probabilidades uniformes del modelo BERT, lo que reduce su aporte en el blend.